In [1]:
using Pkg
Pkg.activate(".")

Pkg.instantiate()
Pkg.status()

###########################################
#= Packages =#
#using Flux, ProgressMeter, Random, Statistics, QuantumOptics, SparseArrays, StatsBase, LinearAlgebra, Revise
#using Zygote, DifferentialEquations, SciMLSensitivity, DiffEqFlux, Optimisers, Compat, PlotlyJS, CSV, DataFrames, BSON
#using Distributions, BSplineKit, DelimitedFiles

#Reduce number of packages to a minimum for the quantumoptics simulations
using ProgressMeter, QuantumOptics, Revise, DelimitedFiles, BSplineKit
##########################################

  Activating project at `~/dev/git/NN_QM/HBAR-qubit_problem`


Status `~/dev/git/NN_QM/HBAR-qubit_problem/Project.toml`
  [c7e460c6] ArgParse v1.2.0
  [fbb218c0] BSON v0.3.9
  [093aae92] BSplineKit v0.19.1
  [336ed68f] CSV v0.10.15
  [34da2185] Compat v4.18.1
  [a93c6f00] DataFrames v1.8.1
  [8bb1440f] DelimitedFiles v1.9.1
⌃ [aae7a2af] DiffEqFlux v4.4.0
  [0c46a032] DifferentialEquations v7.17.0
  [31c24e10] Distributions v0.25.122
  [587475ba] Flux v0.16.5
  [3bd65402] Optimisers v0.4.6
  [f0f68f2c] PlotlyJS v0.18.17
  [92933f4c] ProgressMeter v1.11.0
  [6e0679c1] QuantumOptics v1.2.4
  [295af30f] Revise v3.12.0
  [1ed8b502] SciMLSensitivity v7.90.0
  [10745b16] Statistics v1.11.1
  [2913bbd2] StatsBase v0.34.7
  [e88e6eb3] Zygote v0.7.10
  [37e2e46d] LinearAlgebra v1.11.0
  [9a3f8284] Random v1.11.0
  [2f01184e] SparseArrays v1.11.0
Info Packages marked with ⌃ have new versions available and may be upgradable.


Spin-flip Hamiltonian:
$$ \mathcal{H} = \frac{\Delta_0}{2} \sigma_z + g\left(\sigma_+ a + \sigma_- a^\dagger \right) $$

Swap Hamiltonian

$$ \mathcal{H} = g\left(\sigma_+ a + \sigma_- a^\dagger \right) $$
.

#### Francesco's idea:
1. Chu's protocol works well when $g \ll \Delta = \omega_q - \omega_m$ (weakly-interacting regime) -> Idea is to simulate Chu's protocol and show good infidelities;

2. As soon $g\simeq\Delta$ or $g\gg\Delta$ (strongly-interacting regime), Chu's protocol should become inefficient to produce the FL -> here, we can improve Chu's protocol, by using a NN approach to predict the optimal pulse durations (ideally, we should get smaller infidelities than Chu's ones)


#### What to do:
1. Simulate Chu's protocol with (weakly-interacting regime) and without (strongly-interacting regime) correction, and show the infidelities;

2. Predict the best pulse duration through the NN method with and without correction, and compare the obtained infidelities with Chu's ones (should be lower in the first case, larger in the second case)

3. The critical parameters affecting the working regime are $g$, $\Delta = \omega_q - \omega_m$.

In [2]:
################################################
  #= PRELIMINARY DEFINITIONS AND DYNAMICS =#
################################################
N_mech = 10; #Cut-off Fock basis for the mech resonator

In [3]:
using Revise
include("../definitions.jl")

In [4]:
include("HBAR-qubit_problem.jl")
include("../ML_QM_library.jl")

execute_problem_NN (generic function with 1 method)

In [5]:
using Printf

In [6]:
function save_vector_to_csv(vector, filename::String; header="Header")
    #Write vector to file, with new line as delimiter
    writedlm(filename, [header;vector], "\n")
end


save_vector_to_csv (generic function with 2 methods)

In [7]:

#Mechanical bath
γm_abs = 2.5; #dissipation rate
#Qubit bath
κ_abs = 19; #dissipation rate
κϕ_abs = 21; #dephasing rate
g_abs = 258 ; #JC coupling rate

In [ ]:
#dt = 1e-2; #dt of time integration

## Chu's protocol, varying $g$

In [9]:
N_steps = 2

#[0.01, 0.05], [0.01, 0.05, 0.1, 0.15, 0.2]
#g_rels =  [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
ΩR_rels = [0.01]#, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
g_rels =  [0.01]#, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
#ΩR_rels = [0.8, 0.85, 0.9, 0.95, 1.0]

time_evolutions = [:master_dynamic]#[:schroedinger_dynamic, :master_dynamic]
corrections = [:correction_on]

solution_final = nothing
time_final = nothing

#steps = collect(1:N_steps)
#infidelities_nocorr = []
for time_evo in time_evolutions
    for corr in corrections 
        for ΩR_rel in ΩR_rels
            println(ΩR_rel)
            for g_rel in g_rels 
                
                Δ0_abs = (g_abs/g_rel)
                
                global γm = γm_abs/Δ0_abs
                global κ = κ_abs/Δ0_abs
                global κϕ = κϕ_abs/Δ0_abs
                global g = g_abs/Δ0_abs # same as g_rel (by definition)
                global Δ0 = Δ0_abs/Δ0_abs # for consistency
                
                ΩR = abs(ΩR_rel*Δ0) # also same as ΩR_rel by definition
                
                type_of_dynamics = time_evo
                type_of_correction = corr

                pulse_parameters = [[π / ΩR, π / (2*g*sqrt(n))] for n in 1:N_steps]
                initial_state = tensor(spindown(qub.basis),fockstate(mech_res.basis, 0))
                
                if time_evo ==:schroedinger_dynamic
                    target_states = [tensor(spindown(qub.basis),fockstate(mech_res.basis, n)) for n in 1:N_steps]
                elseif time_evo ==:master_dynamic
                    target_states = [dm(tensor(spindown(qub.basis),fockstate(mech_res.basis, n))) for n in 1:N_steps]
                else 
                    error("Illegal time_evo option")
                end
                
                chu_protocol_features1 = fl_1step_features(
                    [0.0],
                    target_states, 
                    [0], 
                    0, 
                    type_of_correction
                )
                
                time_final, solution_final, infidelities_ = dynamics_n_steps_FL(N_steps, 
                    initial_state,
                    pulse_parameters, 
                    type_of_dynamics, 
                    chu_protocol_features1
                ) 

                g_dir = replace("g_$(@sprintf("%.2f", g_rel))", "." => "_")
                
                #save_vector_to_csv(infidelities_, "relative_Chu_g$(g_rel)D_A$(ΩR_rel)D_$(corr)_$(time_evo).csv", header = "Infidelities")

                #save_vector_to_csv(infidelities_, "./data/simple_ladder/$(time_evo)/$(corr)/$(g_dir)/Chu_g$(g_rel)D_A$(ΩR_rel)D_$(corr)_$(time_evo).csv", header = "Infidelities")
            end
        end
    end
end


0.01


In [11]:
using Plots

In [20]:
plot(time_final, expect(one(qub.basis)⊗n, solution_final))

LoadError: MethodError: no method matching tensor(::Operator{SpinBasis{1//2, Int64}, SpinBasis{1//2, Int64}, LinearAlgebra.Diagonal{ComplexF64, FillArrays.Ones{ComplexF64, 1, Tuple{Base.OneTo{Int64}}}}}, ::Int64)
The function `tensor` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  tensor(::AbstractOperator{B1, B2}, [91m::LazyTensor{B3, B4}[39m) where {B1, B2, B3, B4}
[0m[90m   @[39m [35mQuantumOpticsBase[39m [90m~/.julia/packages/QuantumOpticsBase/u7a0b/src/[39m[90m[4moperators_lazytensor.jl:123[24m[39m
[0m  tensor(::AbstractOperator)
[0m[90m   @[39m [36mQuantumInterface[39m [90m~/.julia/packages/QuantumInterface/QaBbC/src/[39m[90m[4mtensor.jl:7[24m[39m
[0m  tensor(::EyeOpType, [91m::SparseOpType[39m)
[0m[90m   @[39m [35mQuantumOpticsBase[39m [90m~/.julia/packages/QuantumOpticsBase/u7a0b/src/[39m[90m[4moperators_sparse.jl:110[24m[39m
[0m  ...


In [15]:
mech.basis

LoadError: UndefVarError: `mech` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [18]:
mech_res.basis

Fock(cutoff=10)

In [ ]:
replace("g_$(@sprintf("%.2f", 0.15))", "." => "_")

In [ ]:
ΩR_rel

In [ ]:
N_steps = 10

g_rels = [0.2]#, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
ΩR_rels = [0.2]#, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
time_evolutions = [:schroedinger_dynamic]#[:schroedinger_dynamic, :master_dynamic]
corrections = [:correction_on]

sol_fin = nothing
#steps = collect(1:N_steps)
#infidelities_nocorr = []
for time_evo in time_evolutions
    for corr in corrections 
        for ΩR_rel in ΩR_rels
            for g_rel in g_rels 
                
                global Δ0 = (g/g_rel)
                ΩR = abs(ΩR_rel*Δ0) # These are already scaled through g

                
                type_of_dynamics = time_evo
                type_of_correction = corr

                pulse_parameters = [[π / ΩR, π / (2*g*sqrt(n))] for n in 1:N_steps]
                initial_state = tensor(spindown(qub.basis),fockstate(mech_res.basis, 0))
                
                if time_evo ==:schroedinger_dynamic
                    target_states = [tensor(spindown(qub.basis),fockstate(mech_res.basis, n)) for n in 1:N_steps]
                elseif time_evo ==:master_dynamic
                    target_states = [dm(tensor(spindown(qub.basis),fockstate(mech_res.basis, n))) for n in 1:N_steps]
                else 
                    error("Illegal time_evo option")
                end
                
                chu_protocol_features1 = fl_1step_features(
                    [0.0],
                    target_states, 
                    [0], 
                    0, 
                    type_of_correction
                )
                
                time_final, solution_final, infidelities_ = dynamics_n_steps_FL(N_steps, 
                    initial_state,
                    pulse_parameters, 
                    type_of_dynamics, 
                    chu_protocol_features1
                ) 

                sol_fin = solution_final
                #push!(infidelities_nocorr, infidelities_)
                g_dir = replace("g_$(@sprintf("%.2f", g_rel))", "." => "_")
                save_vector_to_csv(infidelities_, "./data/simple_ladder/$(time_evo)/$(corr)/$(g_dir)/Chu_g$(g_rel)D_A$(ΩR_rel)D_$(corr)_$(time_evo).csv", header = "Infidelities")
            end
        end
    end
end


In [ ]:
$(@sprintf("%.2f", g))